# Notebook 01 — Data Download & Exploration
**Week 1 Task:** Download 5yr OHLCV for all Nifty 50 stocks and explore the data.


In [2]:
import sys
import os

# Move working directory to project root
os.chdir('..')
sys.path.append('.')

import pandas as pd
import matplotlib.pyplot as plt
from src.data.download import download_all, load_symbol, load_all_symbols

print('Imports OK')
print('Working dir:', os.getcwd())

Imports OK
Working dir: c:\Users\Aryan\Desktop\everything\PROJECTS\200%BOT\ai-trading-bot\ai-trading-bot


## Step 1 — Download all Nifty 50 data

In [11]:
pip install jugaad-data


   ------------------------------ --------- 3/4 [jugaad-data]
   ---------------------------------------- 4/4 [jugaad-data]

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Inspect a single stock

In [ ]:
import os
files = os.listdir('data/raw')
csv_files = [f for f in files if f.endswith('.NS.csv')]
print(f"NS files: {len(csv_files)}")
print(sorted(csv_files)[:5])

# Load one stock and check it
df = pd.read_csv('data/raw/RELIANCE.NS.csv')
print(f"\nRELIANCE shape: {df.shape}")
print(df.columns.tolist())
print(df.tail(3))

In [4]:
df = load_symbol('RELIANCE.NS')
print(df.shape)
df.tail(10)

FileNotFoundError: data/raw/RELIANCE.NS.csv not found. Run download_all() first.

In [ ]:
# Plot closing price
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(df.index, df['Close'], label='Close Price', color='steelblue')
ax1.set_title('RELIANCE — 5yr Close Price')
ax1.legend()

ax2.bar(df.index, df['Volume'], color='orange', alpha=0.6, label='Volume')
ax2.set_title('Volume')
ax2.legend()

plt.tight_layout()
plt.show()

## Step 3 — Check for missing data

In [ ]:
all_data = load_all_symbols()

summary = []
for symbol, df in all_data.items():
    summary.append({
        'symbol': symbol,
        'rows': len(df),
        'start': df.index[0].date(),
        'end': df.index[-1].date(),
        'null_pct': round(df.isnull().sum().sum() / df.size * 100, 2)
    })

summary_df = pd.DataFrame(summary)
print(summary_df.sort_values('rows').head(10))  # show shortest datasets

## Step 4 — Check market regime today

In [ ]:
from src.data.regime_detector import get_nifty_regime
import pprint

regime = get_nifty_regime()
pprint.pprint(regime)

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

os.chdir('C:\\Users\\Aryan\\Desktop\\everything\\PROJECTS\\200%BOT\\ai-trading-bot\\ai-trading-bot')
sys.path.append('.')

# Check files
files = os.listdir('data/raw')
csv_files = [f for f in files if f.endswith('.NS.csv')]
print(f"NS files: {len(csv_files)}")
print(sorted(csv_files)[:5])

# Load one stock
df = pd.read_csv('data/raw/RELIANCE.NS.csv')
print(f"\nRELIANCE shape: {df.shape}")
print(df.columns.tolist())
print(df.tail(3))

NS files: 52
['ADANIPORTS.NS.csv', 'ASIANPAINT.NS.csv', 'AXISBANK.NS.csv', 'BAJAJ-AUTO.NS.csv', 'BAJAJFINSV.NS.csv']

RELIANCE shape: (5306, 15)
['Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble']
            Date    Symbol Series  Prev Close     Open    High      Low  \
5303  2021-04-28  RELIANCE     EQ     1988.65  1997.85  2008.0  1980.15   
5304  2021-04-29  RELIANCE     EQ     1997.30  2022.90  2044.5  2007.30   
5305  2021-04-30  RELIANCE     EQ     2024.05  2008.50  2036.0  1987.55   

         Last    Close     VWAP   Volume      Turnover    Trades  \
5303  1993.15  1997.30  1997.60  7902002  1.578508e+15  247331.0   
5304  2020.00  2024.05  2024.21  8035915  1.626634e+15  213153.0   
5305  1995.90  1994.50  2010.20  9150974  1.839532e+15  288687.0   

      Deliverable Volume  %Deliverble  
5303           3921560.0       0.4963  
5304           2834103.0       0.3527  
5

In [2]:
import os
import pandas as pd

def load_and_clean(symbol_ns):
    df = pd.read_csv(f'data/raw/{symbol_ns}', parse_dates=['Date'])
    
    # Keep only what we need
    df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df = df.sort_values('Date').set_index('Date')
    df = df.dropna()
    
    # Filter to last 5 years of data (2016 onwards is plenty)
    df = df[df.index >= '2016-01-01']
    
    return df

# Test with RELIANCE
df = load_and_clean('RELIANCE.NS.csv')
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
print(df.tail(3))

Shape: (1318, 5)
Date range: 2016-01-01 → 2021-04-30
               Open    High      Low    Close   Volume
Date                                                  
2021-04-28  1997.85  2008.0  1980.15  1997.30  7902002
2021-04-29  2022.90  2044.5  2007.30  2024.05  8035915
2021-04-30  2008.50  2036.0  1987.55  1994.50  9150974


In [3]:
os.makedirs('data/processed', exist_ok=True)

csv_files = [f for f in os.listdir('data/raw') if f.endswith('.NS.csv')]
success = 0

for f in csv_files:
    try:
        df = load_and_clean(f)
        if len(df) > 100:  # skip if too little data
            df.to_csv(f'data/processed/{f}')
            success += 1
            print(f"✅ {f} — {len(df)} rows")
    except Exception as e:
        print(f"❌ {f} — {e}")

print(f"\nDone — {success}/{len(csv_files)} cleaned files saved to data/processed/")

✅ ADANIPORTS.NS.csv — 1318 rows
✅ ASIANPAINT.NS.csv — 1318 rows
✅ AXISBANK.NS.csv — 1318 rows
✅ BAJAJ-AUTO.NS.csv — 1318 rows
✅ BAJAJFINSV.NS.csv — 1318 rows
✅ BAJFINANCE.NS.csv — 1318 rows
✅ BHARTIARTL.NS.csv — 1318 rows
✅ BPCL.NS.csv — 1318 rows
✅ BRITANNIA.NS.csv — 1318 rows
✅ CIPLA.NS.csv — 1318 rows
✅ COALINDIA.NS.csv — 1318 rows
✅ DRREDDY.NS.csv — 1318 rows
✅ EICHERMOT.NS.csv — 1318 rows
✅ GAIL.NS.csv — 1318 rows
✅ GRASIM.NS.csv — 1318 rows
✅ HCLTECH.NS.csv — 1318 rows
✅ HDFC.NS.csv — 1318 rows
✅ HDFCBANK.NS.csv — 1318 rows
✅ HEROMOTOCO.NS.csv — 1318 rows
✅ HINDALCO.NS.csv — 1318 rows
✅ HINDUNILVR.NS.csv — 1318 rows
✅ ICICIBANK.NS.csv — 1318 rows
✅ INDUSINDBK.NS.csv — 1318 rows
✅ INFY.NS.csv — 1318 rows
✅ IOC.NS.csv — 1318 rows
✅ ITC.NS.csv — 1318 rows
✅ JSWSTEEL.NS.csv — 1318 rows
✅ KOTAKBANK.NS.csv — 1318 rows
✅ LT.NS.csv — 1318 rows
✅ MARUTI.NS.csv — 1318 rows
✅ MM.NS.csv — 1318 rows
✅ NESTLEIND.NS.csv — 1318 rows
✅ NIFTY50_all.NS.csv — 64582 rows
✅ NTPC.NS.csv — 1318 rows
✅ O

In [4]:
import pandas as pd
import numpy as np
import os

def add_features(df):
    # ── Trend indicators ──
    df['MA_20']  = df['Close'].rolling(20).mean()
    df['MA_50']  = df['Close'].rolling(50).mean()
    df['MA_200'] = df['Close'].rolling(200).mean()
    
    # Price vs MA signals
    df['above_MA20']  = (df['Close'] > df['MA_20']).astype(int)
    df['above_MA200'] = (df['Close'] > df['MA_200']).astype(int)
    
    # ── Momentum indicators ──
    # RSI
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))
    
    # MACD
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']
    
    # ── Volatility indicators ──
    # Bollinger Bands
    bb_mid         = df['Close'].rolling(20).mean()
    bb_std         = df['Close'].rolling(20).std()
    df['BB_upper'] = bb_mid + 2 * bb_std
    df['BB_lower'] = bb_mid - 2 * bb_std
    df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / bb_mid
    df['BB_pos']   = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])
    
    # ATR (Average True Range)
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift()).abs(),
        (df['Low']  - df['Close'].shift()).abs()
    ], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(14).mean()
    
    # ── Volume indicators ──
    df['Volume_MA20'] = df['Volume'].rolling(20).mean()
    df['Volume_ratio'] = df['Volume'] / df['Volume_MA20']  # >1 = high volume
    
    # ── Price action ──
    df['Daily_return']   = df['Close'].pct_change()
    df['High_Low_range'] = (df['High'] - df['Low']) / df['Close']
    
    # ── Target variable ──
    # 1 = price goes UP next day, 0 = goes down
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    return df

# Apply to all stocks
os.makedirs('data/processed/features', exist_ok=True)
processed = [f for f in os.listdir('data/processed') if f.endswith('.NS.csv')]

success = 0
for f in processed:
    try:
        df = pd.read_csv(f'data/processed/{f}', index_col='Date', parse_dates=True)
        df = add_features(df)
        df = df.dropna()  # remove rows where indicators aren't ready yet
        df.to_csv(f'data/processed/features/{f}')
        success += 1
    except Exception as e:
        print(f"❌ {f} — {e}")

print(f"✅ Features added to {success}/{len(processed)} stocks")

# Show what features we have
sample = pd.read_csv('data/processed/features/RELIANCE.NS.csv', index_col='Date', parse_dates=True)
print(f"\nFeature count: {len(sample.columns)}")
print(f"Rows after dropna: {len(sample)}")
print(f"\nColumns:\n{sample.columns.tolist()}")

✅ Features added to 50/50 stocks

Feature count: 24
Rows after dropna: 1119

Columns:
['Open', 'High', 'Low', 'Close', 'Volume', 'MA_20', 'MA_50', 'MA_200', 'above_MA20', 'above_MA200', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_lower', 'BB_width', 'BB_pos', 'ATR', 'Volume_MA20', 'Volume_ratio', 'Daily_return', 'High_Low_range', 'Target']
